# QQA 00 – Colab quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuma-Ichikawa/QQA4CO/blob/main/examples/00_colab_quickstart.ipynb)

One-notebook tour of every built-in problem, designed to run on a free Google Colab CPU runtime.

In [ ]:
# Install QQA on Google Colab (no-op if already installed locally)
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('qqa') is None:
    subprocess.check_call(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            'qqa @ git+https://github.com/Yuma-Ichikawa/QQA4CO.git',
        ]
    )

This notebook walks through every problem family shipped with QQA on Google Colab. It installs `qqa` from GitHub, detects CUDA if available, and runs a small `qqa.anneal` job per problem with an inline `viz.plot_history` / `viz.plot_best_trajectory` figure. The whole notebook finishes in ~2 minutes on a free CPU Colab runtime and ~30s on a GPU runtime.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import torch

import qqa
from qqa import visualization as viz

qqa.fix_seed(0)
print('QQA version:', qqa.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. Maximum Independent Set

In [ ]:
g = nx.random_regular_graph(d=3, n=50, seed=0)
problem = qqa.MaximumIndependentSet(g, penalty=2, device=device)
r = qqa.anneal(problem, sol_size=64, num_epochs=1000, device=device, verbose=False)
print(f'MIS size >= {-int(r.best_obj)}  ({r.runtime:.2f}s)')
viz.plot_history(r, show=False);

## 2. Graph coloring (K=3)

In [ ]:
g = nx.random_regular_graph(d=3, n=40, seed=0)
problem = qqa.Coloring(g, num_category=3, device=device)
r = qqa.anneal(problem, sol_size=64, num_epochs=1500, device=device, verbose=False)
print(f'conflicts: {int(round(r.best_obj))}')
viz.plot_best_trajectory(r, show=False);

## 3. Max-Cut

In [ ]:
g = nx.erdos_renyi_graph(n=40, p=0.2, seed=0)
problem = qqa.MaxCut(g, device=device)
r = qqa.anneal(problem, sol_size=64, num_epochs=1000, device=device, verbose=False)
print(f'cut value >= {-float(r.best_obj):.2f}')
viz.plot_history(r, show=False);

## 4. 1D Ising ferromagnet

In [ ]:
problem = qqa.Ising1D(N=32, J=1.0, periodic=True, device=device)
r = qqa.anneal(problem, sol_size=64, num_epochs=600, device=device, verbose=False)
print(f'E = {float(r.best_obj):.3f}  (target: -32)')
viz.plot_best_trajectory(r, show=False);

## 5. Edwards–Anderson 3D spin glass

In [ ]:
problem = qqa.EdwardsAnderson(L=4, dim=3, seed=0, device=device)
r = qqa.anneal(problem, sol_size=128, num_epochs=1500, device=device, verbose=False)
print(f'E / N = {float(r.best_obj) / problem.num_spins:.4f}')
viz.plot_history(r, show=False);

## 6. Sherrington–Kirkpatrick

In [ ]:
problem = qqa.SherringtonKirkpatrick(N=100, seed=0, device=device)
r = qqa.anneal(problem, sol_size=128, num_epochs=1500, device=device, verbose=False)
print(f'e_0 = {float(r.best_obj) / 100:.4f}  (Parisi: -0.7632)')
viz.plot_best_trajectory(r, show=False);

## 7. Binary perceptron

In [ ]:
problem = qqa.BinaryPerceptron(N=30, alpha=0.5, seed=0, sharpness=10.0, device=device)
r = qqa.anneal(problem, sol_size=128, num_epochs=1500, device=device, verbose=False)
s_best = problem.relaxation.project(r.best_sol).unsqueeze(0)
print(f'min errors = {int(problem.error_count(s_best).min())}')
viz.plot_best_trajectory(r, show=False);

## 8. Hopfield memory

In [ ]:
problem = qqa.HopfieldMemory(N=64, patterns=3, seed=0, device=device)
r = qqa.anneal(problem, sol_size=128, num_epochs=1000, device=device, verbose=False)
s_best = problem.relaxation.project(r.best_sol).unsqueeze(0)
overlap = problem.overlap(s_best).abs().max().item()
print(f'max overlap with stored pattern: {overlap:.3f}')
viz.plot_history(r, show=False);

## 9. Parallel MIS (`MaximumIndependentSetInstance`)

In [ ]:
N, degrees = 60, [2, 3, 4, 5]
graphs = [nx.random_regular_graph(d=d, n=N, seed=d) for d in degrees]
problem = qqa.MaximumIndependentSetInstance(graphs, max_node=N, penalty=2, device=device)
r = qqa.anneal(problem, sol_size=64, num_epochs=800, device=device, verbose=False)
for d, obj in zip(degrees, r.best_obj, strict=False):
    print(f'  degree={d}: MIS >= {-int(round(float(obj)))}')

## Custom loss via `UserProblem`

In [ ]:
import torch

N = 40
g = torch.Generator().manual_seed(0)
J = torch.randn(N, N, generator=g) / (N ** 0.5)
J = (J + J.T) / 2
J.fill_diagonal_(0.0)
problem = qqa.UserProblem(
    num_vars=N, variable_kind='spin',
    loss_fn=lambda s: -0.5 * torch.einsum('bi,ij,bj->b', s, J, s),
)
r = qqa.anneal(problem, sol_size=128, num_epochs=1000, verbose=False)
print(f'custom spin-glass e_0 = {float(r.best_obj) / N:.4f}')